# DAG Ejecutado

In [ ]:
#Librerías:
import re
import json
import os

from bs4 import BeautifulSoup
import pandas as pd
import urllib.request as urlb
import py7zr
from sklearn.model_selection import train_test_split
import sqlite3


In [60]:
DB_PATH = '/tmp/airflow_etl.db'  # Base de datos local para Airflow
pathToData = "../data/raw/"

def GetData():
    """
    Esta función únicamente se ejecuta entera una vez o en caso de que se borren los datos. 
    Se encarga de descargar, descomprimir y pasar a pickle los datos formato xml.
    También crea un db en DB_PATH que se usará para el manejo de los dataframes.
    Devuelve el nombre de los conjuntos que se usarán separados en train y test.
    """
    url = "https://archive.org/download/stackexchange/english.stackexchange.com.7z"
    files = ["Users.xml", "Posts.xml"]
    archive = 0
    
    for file in files:
        if file not in os.listdir(pathToData):
            if "data.7z" not in os.listdir(pathToData):
                urlb.urlretrieve(url, str(pathToData) + "data.7z")
            archive = py7zr.SevenZipFile(str(pathToData) + "data.7z", 'r')
            archive.extract(path=pathToData, targets=[file])
            archive.reset()
    
    if archive != 0:
        archive.close()
    
    Users = pd.read_pickle(os.path.abspath(str(pathToData) + "Users.pkl"))
    Posts = pd.read_pickle(os.path.abspath(str(pathToData) + "Posts.pkl"))

    Users_Train, Users_Test = train_test_split(Users, test_size=0.3)
    Posts_Train, Posts_Test = train_test_split(Posts, test_size=0.3)

    conn = sqlite3.connect(DB_PATH)
    Users_Train.to_sql("Users_Train", conn, if_exists="replace", index=False)
    Users_Test.to_sql("Users_Test", conn, if_exists="replace", index=False)
    Posts_Train.to_sql("Posts_Train", conn, if_exists="replace", index=False)
    Posts_Test.to_sql("Posts_Test", conn, if_exists="replace", index=False)
    conn.close()

    return {
        "Users_Train": "Users_Train",
        "Users_Test": "Users_Test",
        "Posts_Train": "Posts_Train",
        "Posts_Test": "Posts_Test"
    }


def RemoveNull(table_name: str):
    """
    Elimina las filas donde AccountId es nulo y elimina la columna AccountId del dataframe Users.
    También convierte todos los tipos de Null a uno único: pd.NA (O eso se supone puesto que en la práctica por alguna razón no pasa)
    """        
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    df = df.dropna(subset=['AccountId'])
    df = df.drop(columns=['AccountId'], errors='ignore')
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return table_name


def TransformWebsiteURL(table_name: str):   #TICK
    """
    Modifica el campo WebsiteUrl del dataframe de Users a un campo binario que marca si tiene alguna URL en su perfil o no
    """
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    df["WebsiteUrl"] = df["WebsiteUrl"].notna()
    df = df.rename(columns={"WebsiteUrl": "HasUrl"})
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return table_name


def ConvertDates(table_name: str):          #TICK
    """
    Convierte las columnas que deberían ser fechas de string a datetime,
    omitiendo los milisegundos.
    """
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    if "Users" in table_name:
        campos = ['CreationDate', 'LastAccessDate']
    else:
        campos = ['CreationDate','LastActivityDate', 'LastEditDate', 'ClosedDate', 'CommunityOwnedDate']
    
    for col in campos:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).astype('datetime64[s]')
            
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return table_name


def HTML_to_Text(table_name: str):          #TICK
    """
    Convierte las variables que contienen HTML a un string donde solo se tiene el texto contenido en ese HTML
    """
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    if 'Users' in table_name:
        df['AboutMe'] = df['AboutMe'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text().strip() if isinstance(x, str) and x.strip() else pd.NA)
        df["AboutMe"] = df["AboutMe"].replace("", pd.NA)
    if 'Posts' in table_name:
        df['Body'] = df['Body'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text().lower() if isinstance(x, str) and x.strip() else pd.NA)
        df['Body'] = df['Body'].replace("", pd.NA)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return table_name


def Country_Location(table_name:str):
    """
    Carga formas equivalentes de escribir un país, para poder detectar la nacionalidad de un usuario.
    Modifica el campo "Location" para que sea únicamente el país. En caso de no reconocer ninguno de la lista, lo vuelve pd.NA.
    De igual forma también crea una nueva variable binaria llamada HasLocation que determina si tiene algún país en su campo Location o no.
    """
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    
    with open("../data/paisesDict.json", "r", encoding="utf-8") as archivo:
        countries = json.load(archivo)

    pattern_dict = {}
    for country, variations in countries.items():
        safe_variations = [re.escape(v) for v in variations]  # Escapar caracteres especiales
        pattern = r'\b(' + '|'.join(safe_variations) + r')\b'  # \b asegura coincidencia exacta
        pattern_dict[country] = re.compile(pattern, re.IGNORECASE)

        def get_country(location):
            if not isinstance(location, str):
                return pd.NA  # Si no es una cadena, devolvemos NA

            for country, pattern in pattern_dict.items():
                if pattern.search(location):
                    return country  # Retorna el primer país coincidente

            return pd.NA  # Si no hay coincidencia


    df["Location"] = df["Location"].apply(get_country)
    df["HasLocation"] = df["Location"].notna().astype(bool)
    
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return table_name


def Parse_Tags(table_name):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    
    df['Tags'] = df['Tags'].fillna('')  # Rellenar valores NaN con cadenas vacías
    df['Tags'] = df['Tags'].apply(lambda x: x.split('|') if x else [])

    df.to_sql(table_name,conn,if_exists='replace',index = False)
    conn.close()
    return table_name



def LoadData(table_name: str):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    # Guardar el DataFrame en un archivo CSV
    csv_path = f"../data/processed/{table_name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"DataFrame guardado en {csv_path}")


In [61]:
# Obtener los nombres de las tablas
data = GetData()
users_train_table = data["Users_Train"]
posts_train_table = data["Posts_Train"]

# Procesar Users_Train
users_train_table = RemoveNull(table_name=users_train_table)
users_train_table = TransformWebsiteURL(table_name=users_train_table)
users_train_table = ConvertDates(table_name=users_train_table)
users_train_table = HTML_to_Text(table_name=users_train_table)
users_train_table = Country_Location(table_name=users_train_table)
LoadData(table_name=users_train_table)

# Procesar Posts_Train
posts_train_table = ConvertDates(table_name=posts_train_table)
posts_train_table = HTML_to_Text(table_name=posts_train_table)
#posts_train_table = Parse_Tags(table_name=posts_train_table)
LoadData(table_name=posts_train_table)

/tmp/ipykernel_3989/2632822997.py:102: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  df['AboutMe'] = df['AboutMe'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text().strip() if isinstance(x, str) and x.strip() 

DataFrame guardado en ../data/processed/Users_Train.csv
DataFrame guardado en ../data/processed/Posts_Train.csv
